# LeetCode 384 — Shuffle an Array bằng Python

Link đề: https://leetcode.com/problems/shuffle-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Math, Design, Randomized

Skeleton LeetCode:

```python
class Solution:

    def __init__(self, nums: List[int]):
        ...

    def reset(self) -> List[int]:
        ...

    def shuffle(self) -> List[int]:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(nums)
# param_1 = obj.reset()
# param_2 = obj.shuffle()
```

Ghi chú: bài **thiết kế lớp + ngẫu nhiên**. Đáp án không cố định, nên notebook test bằng **tính chất** (kết quả là hoán vị của mảng gốc, `reset` trả đúng mảng gốc) và **thống kê/vét cạn có seed** (mọi hoán vị xuất hiện với xác suất bằng nhau).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế lớp trộn mảng sao cho **mọi hoán vị** của `nums` có **xác suất như nhau**, kèm `reset()` trả về mảng gốc.
2. Nắm các công cụ Python: sao chép list (`nums[:]`, `list(nums)`), hoán đổi `a[i], a[j] = a[j], a[i]`, `random.randrange`/`randint`, `random.seed`, `itertools.product`/`permutations` để **liệt kê mọi khả năng**, và khác biệt giữa **alias** (hai tên cùng một list) với **bản sao**.
3. Đi từ **bốc ngẫu nhiên từ "túi" rồi bỏ ra** → **sắp xếp theo khóa ngẫu nhiên** → **Fisher–Yates** → **Fisher–Yates "inside-out"**.
4. Hiểu mấu chốt: có `n!` hoán vị, nên thuật toán phải có **đúng `n·(n-1)·…·1 = n!` chuỗi lựa chọn ngẫu nhiên đồng khả năng**, mỗi chuỗi cho một hoán vị khác nhau — Fisher–Yates làm đúng điều đó.
5. Tránh các lỗi kinh điển: hoán đổi với **mọi** vị trí (`n^n` kết quả, lệch), nhầm biên làm thành thuật toán Sattolo (không bao giờ giữ nguyên vị trí), trộn tại chỗ làm mất mảng gốc, và so sánh ngẫu nhiên bằng `cmp` ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy cài lớp `Solution`:

- `Solution(nums)`: khởi tạo với mảng `nums`.
- `reset()`: đặt lại về **cấu hình ban đầu** và trả về mảng đó.
- `shuffle()`: trả về **một hoán vị ngẫu nhiên** của mảng. **Mọi hoán vị phải có xác suất bằng nhau.**

**Ví dụ:**

```
Input
["Solution", "shuffle", "reset", "shuffle"]
[[[1, 2, 3]], [], [], []]
Output
[null, [3, 1, 2], [1, 2, 3], [1, 3, 2]]
```

(`shuffle` có thể trả bất kỳ hoán vị nào của `[1, 2, 3]`; `reset` trả đúng `[1, 2, 3]`.)

**Ràng buộc:** `1 <= nums.length <= 50`; `-10^6 <= nums[i] <= 10^6`; mọi phần tử **khác nhau**; tối đa `10^4` lần gọi `reset`/`shuffle`.

In [1]:
import random
import time
from collections import Counter
from functools import cmp_to_key
from itertools import permutations, product
from math import factorial
from typing import List


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp: ops[0] là hàm tạo."""
    obj, out = None, []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops = ["Solution", "shuffle", "reset", "shuffle"]
args = [[[1, 2, 3]], [], [], []]
print(ops, args)
print("số hoán vị của [1,2,3]:", factorial(3))

['Solution', 'shuffle', 'reset', 'shuffle'] [[[1, 2, 3]], [], [], []]
số hoán vị của [1,2,3]: 6


## 3. Kiến thức nền

### 3.1. Hoán vị và `n!`

Một **hoán vị** của `[1, 2, 3]` là cách sắp xếp lại ba số. Có `3! = 6` hoán vị. Tổng quát `n!` — tăng cực nhanh: `10! = 3 628 800`, `50!` là số khổng lồ (vì thế ta **không thể** vét cạn mọi hoán vị để chọn ngẫu nhiên khi `n` lớn).

In [2]:
perms = list(permutations([1, 2, 3]))
print(len(perms), perms)
print([factorial(n) for n in range(1, 8)])

6 [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
[1, 2, 6, 24, 120, 720, 5040]


### 3.2. Sao chép list: alias và bản sao

`b = a` **không** sao chép: `a` và `b` là **hai tên của cùng một list**; sửa qua một tên thì thấy ở tên kia. Muốn bản sao độc lập (shallow copy): `a[:]` hoặc `list(a)`.

In [3]:
a = [1, 2, 3]
alias = a
copy = a[:]
alias[0] = 99
print(a, alias, copy)         # [99,2,3] [99,2,3] [1,2,3]
print(alias is a, copy is a)  # True False

[99, 2, 3] [99, 2, 3] [1, 2, 3]
True False


### 3.3. Hoán đổi hai phần tử

`a[i], a[j] = a[j], a[i]` đổi chỗ **an toàn** (vế phải được tính xong trước khi gán). Khi `i == j` thì không đổi gì.

In [4]:
a = [10, 20, 30, 40]
a[0], a[3] = a[3], a[0]
print(a)           # [40, 20, 30, 10]
a[1], a[1] = a[1], a[1]
print(a)           # không đổi

[40, 20, 30, 10]
[40, 20, 30, 10]


### 3.4. `random`: biên của `randrange` và `randint`

- `random.randrange(a, b)`: đều trong `[a, b)` — **không** gồm `b`.
- `random.randint(a, b)`: đều trong `[a, b]` — **gồm** `b`.
- `random.seed(x)`: cố định dãy ngẫu nhiên.
- `random.random()`: số thực đều trong `[0, 1)`.

In [5]:
random.seed(3)
print({random.randrange(2, 5) for _ in range(300)})    # {2, 3, 4}
print({random.randint(2, 5) for _ in range(300)})      # {2, 3, 4, 5}

{2, 3, 4}
{2, 3, 4, 5}


### 3.5. Sắp xếp theo khóa `key=`

`sorted(seq, key=f)` sắp xếp theo giá trị `f(x)`. Nếu `f` cho mỗi phần tử một **số ngẫu nhiên**, ta được thứ tự ngẫu nhiên. `cmp_to_key` dùng để chuyển hàm so sánh hai-đối-số thành `key` (sẽ dùng để minh họa một lỗi ở mục 13).

In [6]:
words = ["bb", "a", "cccc", "ddd"]
print(sorted(words, key=len))          # ['a', 'bb', 'ddd', 'cccc']
random.seed(1)
print(sorted(words, key=lambda _: random.random()))   # thứ tự ngẫu nhiên

['a', 'bb', 'ddd', 'cccc']
['bb', 'ddd', 'cccc', 'a']


### 3.6. Liệt kê mọi kết quả ngẫu nhiên bằng `product`

Một thuật toán ngẫu nhiên thực chất là: "bốc dãy số `r_1, r_2, …` rồi tính toán xác định". Nếu mỗi `r_i` chạy trong một khoảng nhỏ, ta **duyệt hết mọi tổ hợp** bằng `itertools.product` — mỗi tổ hợp đồng khả năng — và đếm ra phân phối **chính xác**, không cần thống kê.

In [7]:
choices = list(product(range(1, 3), range(1, 4)))      # r1 ∈ {1,2}, r2 ∈ {1,2,3}
print(len(choices), choices)

6 [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bạn có một **bộ bài 3 lá** `[1, 2, 3]` và muốn xáo thật đều.

Cách tự nhiên: **chọn lá đầu tiên** ngẫu nhiên trong cả 3 lá (mỗi lá `1/3`), **chọn lá thứ hai** ngẫu nhiên trong 2 lá còn lại (mỗi lá `1/2`), lá cuối thì chỉ còn 1 lá. Số chuỗi lựa chọn: `3 × 2 × 1 = 6` — **đúng bằng số hoán vị**, mỗi chuỗi cho một hoán vị khác nhau, mỗi hoán vị có xác suất `1/3 · 1/2 · 1 = 1/6`.

Tính tay một ví dụ: chọn lá thứ 3 (số `3`) trước, rồi trong `{1, 2}` chọn `1`, cuối cùng `2` ⇒ hoán vị `[3, 1, 2]` — chính là ví dụ của đề.

Fisher–Yates thực hiện **đúng ý đó tại chỗ**: bước `i` chọn một phần tử trong phần **chưa cố định** `a[i..n-1]` rồi đổi lên vị trí `i`.

## 5. Cách 1 — Brute Force / baseline: bốc từ "túi" rồi bỏ ra

**Ý tưởng:** sao chép mảng thành một "túi" `pool`. Lặp đến khi túi rỗng: chọn chỉ số ngẫu nhiên `k = randrange(len(pool))`, **lấy** `pool.pop(k)` đặt vào kết quả.

Đúng chuẩn mô hình ở mục 4 (`n!` chuỗi lựa chọn, đều). Nhưng `pool.pop(k)` ở giữa list phải **dồn các phần tử sau** ⇒ `O(n)` mỗi lần ⇒ cả lần xáo tốn **`O(n²)`**.

In [8]:
class SolutionPool:
    """Cách 1: bốc từ túi. shuffle O(n^2)."""

    def __init__(self, nums):
        self.original = nums[:]

    def reset(self):
        return self.original[:]

    def shuffle(self):
        pool = self.original[:]
        out = []
        while pool:
            k = random.randrange(len(pool))
            out.append(pool.pop(k))          # pop giữa list: O(n)
        return out


random.seed(1)
res = run_ops(SolutionPool, ops, args)
print(res)
assert sorted(res[1]) == [1, 2, 3] and res[2] == [1, 2, 3] and sorted(res[3]) == [1, 2, 3]

[None, [1, 2, 3], [1, 2, 3], [1, 3, 2]]


**Độ phức tạp:** `shuffle` `O(n²)`, bộ nhớ `O(n)`.

**Nút thắt:** `pop` ở giữa tốn `O(n)`. Muốn `O(n)`: thay vì **dồn** phần tử, ta **đổi chỗ** phần tử được chọn với một vị trí biên (đó chính là Fisher–Yates).

## 6. Ý tưởng thuật toán / bước đột phá

Thay vì xóa phần tử khỏi túi, chia **chính mảng** thành hai vùng: `a[0..i-1]` là vùng **đã xếp xong**, `a[i..n-1]` là **túi** còn lại. Bước `i`:

1. Chọn `j` ngẫu nhiên **đều** trong `[i, n-1]` (một phần tử của túi).
2. **Đổi chỗ** `a[i]` và `a[j]` — phần tử vừa chọn nằm ở vị trí `i` (vùng đã xếp), phần tử cũ của `a[i]` về túi.

Không cần dồn, mỗi bước `O(1)` ⇒ tổng `O(n)`.

**Vì sao đều?** Số lượng chuỗi lựa chọn `(j_0, j_1, …, j_{n-1})` là `n · (n-1) · … · 1 = n!`; mỗi chuỗi cho **một hoán vị khác nhau** (vì mỗi bước "cố định" một phần tử rõ ràng) và chiếm xác suất `1/n!`. Ta sẽ **kiểm chứng bằng vét cạn** ở mục 7 và 8.

Một cách nói khác (bất biến): sau bước `i`, phần tử ở vị trí `i` là một phần tử **bất kỳ** trong các phần tử chưa bị cố định, với xác suất đều `1/(n-i)`.

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (bốc từ túi, mục 5), Cách 2 (sắp xếp theo khóa ngẫu nhiên), Cách 3 (Fisher–Yates — **chính thức**), Cách 4 (Fisher–Yates "inside-out").

### Cách 2 — Sắp xếp theo khóa ngẫu nhiên

Giải thích từng đoạn:

1. Mỗi phần tử bốc một khóa `random.random()` độc lập.
2. Sắp xếp theo khóa. Vì các khóa độc lập, cùng phân phối, **mọi thứ tự khóa đồng khả năng** ⇒ mọi hoán vị đều xác suất `1/n!` (khả năng hai khóa trùng nhau là rất nhỏ, bỏ qua).

Chi phí sắp xếp `O(n log n)`. Chú ý: phải gắn **một khóa cố định cho mỗi phần tử** (dùng `key=`), **không** dùng hàm so sánh ngẫu nhiên (xem mục 13).

In [9]:
class SolutionSortKey:
    """Cách 2: sắp xếp theo khóa ngẫu nhiên. shuffle O(n log n)."""

    def __init__(self, nums):
        self.original = nums[:]

    def reset(self):
        return self.original[:]

    def shuffle(self):
        keyed = [(random.random(), x) for x in self.original]    # mỗi phần tử một khóa cố định
        keyed.sort()
        return [x for _, x in keyed]


random.seed(2)
res = run_ops(SolutionSortKey, ops, args)
print(res)
assert sorted(res[1]) == [1, 2, 3] and res[2] == [1, 2, 3]

[None, [3, 2, 1], [1, 2, 3], [1, 3, 2]]


**Code nộp LeetCode cho Cách 2:**

```python
import random

class Solution:
    def __init__(self, nums: List[int]):
        self.original = nums[:]

    def reset(self) -> List[int]:
        return self.original[:]

    def shuffle(self) -> List[int]:
        keyed = [(random.random(), x) for x in self.original]
        keyed.sort()
        return [x for _, x in keyed]
```

**Độ phức tạp:** `shuffle` `O(n log n)`, bộ nhớ `O(n)`.

---

### Cách 3 — Fisher–Yates (Knuth shuffle) — chính thức

Giải thích từng đoạn:

1. Sao chép `a = self.original[:]` (để **không** làm hỏng mảng gốc).
2. `for i in range(n)`: chọn `j = random.randrange(i, n)` (đều trong `[i, n-1]`).
3. Đổi chỗ `a[i], a[j] = a[j], a[i]`.
4. Trả `a`.

(Bước `i = n-1`: `randrange(n-1, n)` luôn `n-1`, không đổi — có thể dừng ở `n-2`; giữ nguyên cho dễ hiểu.)

In [10]:
class SolutionFY:
    """Cách 3 (demo): Fisher-Yates, chọn j trong [i, n-1] rồi đổi chỗ."""

    def __init__(self, nums):
        self.original = nums[:]

    def reset(self):
        return self.original[:]

    def shuffle(self):
        a = self.original[:]
        n = len(a)
        for i in range(n):
            j = random.randrange(i, n)          # j ∈ [i, n-1]
            a[i], a[j] = a[j], a[i]
        return a


random.seed(3)
res = run_ops(SolutionFY, ops, args)
print(res)
assert sorted(res[1]) == [1, 2, 3] and res[2] == [1, 2, 3]

[None, [1, 2, 3], [1, 2, 3], [3, 1, 2]]


**Code nộp LeetCode cho Cách 3:**

```python
import random

class Solution:
    def __init__(self, nums: List[int]):
        self.original = nums[:]

    def reset(self) -> List[int]:
        return self.original[:]

    def shuffle(self) -> List[int]:
        a = self.original[:]
        n = len(a)
        for i in range(n):
            j = random.randrange(i, n)
            a[i], a[j] = a[j], a[i]
        return a
```

**Độ phức tạp:** `shuffle` `O(n)`, bộ nhớ `O(n)` (bản sao để trả về; thuật toán thêm `O(1)`).

---

### Cách 4 — Fisher–Yates "inside-out" (dựng mảng mới từng phần tử)

**Ý tưởng:** không đổi chỗ trong mảng cũ, mà **dựng mảng mới `a`** từng phần tử. Với phần tử thứ `i` của mảng gốc: chọn `j` đều trong `[0, i]`; nếu `j != i`, **chuyển** `a[j]` xuống vị trí `i` rồi đặt phần tử mới vào `a[j]`.

Quy nạp: sau bước `i`, `a[0..i]` là một hoán vị **đều** của `nums[0..i]`. Chèn phần tử mới vào **một trong `i+1` vị trí** đều nhau (vị trí `j` hoặc cuối) giữ nguyên tính đều. Chuỗi lựa chọn: `1 · 2 · 3 · … · n = n!`.

In [11]:
class SolutionInsideOut:
    """Cách 4: Fisher-Yates inside-out (dựng mảng mới)."""

    def __init__(self, nums):
        self.original = nums[:]

    def reset(self):
        return self.original[:]

    def shuffle(self):
        src = self.original
        a = [0] * len(src)
        for i in range(len(src)):
            j = random.randint(0, i)            # j ∈ [0, i]
            if j != i:
                a[i] = a[j]                      # đẩy phần tử cũ xuống vị trí i
            a[j] = src[i]                        # phần tử mới vào vị trí j
        return a


random.seed(4)
res = run_ops(SolutionInsideOut, ops, args)
print(res)
assert sorted(res[1]) == [1, 2, 3] and res[2] == [1, 2, 3]

[None, [3, 2, 1], [1, 2, 3], [3, 2, 1]]


**Code nộp LeetCode cho Cách 4:**

```python
import random

class Solution:
    def __init__(self, nums: List[int]):
        self.original = nums[:]

    def reset(self) -> List[int]:
        return self.original[:]

    def shuffle(self) -> List[int]:
        src = self.original
        a = [0] * len(src)
        for i in range(len(src)):
            j = random.randint(0, i)
            if j != i:
                a[i] = a[j]
            a[j] = src[i]
        return a
```

**Độ phức tạp:** `shuffle` `O(n)`, bộ nhớ `O(n)`.

**Kiểm chứng chính xác (vét cạn):** với mỗi thuật toán ta viết phiên bản **nhận sẵn dãy lựa chọn**, rồi chạy qua **mọi** tổ hợp lựa chọn và đếm phân phối. Nếu mỗi hoán vị xuất hiện **đúng 1 lần** thì xác suất **chính xác `1/n!`**.

In [12]:
def pool_with(nums, ch):          # ch[t] ∈ [0, n - t)
    pool, out = list(nums), []
    for c in ch:
        out.append(pool.pop(c))
    return tuple(out)


def sortkey_with(nums, ch):       # ch = một thứ tự khóa: ch[k] = hạng khóa của phần tử k
    return tuple(x for _, x in sorted(zip(ch, nums)))


def fy_with(nums, ch):            # ch[i] ∈ [i, n-1]
    a = list(nums)
    for i, j in enumerate(ch):
        a[i], a[j] = a[j], a[i]
    return tuple(a)


def inside_out_with(nums, ch):    # ch[i] ∈ [0, i]
    a = [0] * len(nums)
    for i, j in enumerate(ch):
        if j != i:
            a[i] = a[j]
        a[j] = nums[i]
    return tuple(a)


def exact_counts(fn, nums, ranges):
    cnt = Counter()
    for ch in product(*ranges):
        cnt[fn(nums, ch)] += 1
    return cnt


nums = [1, 2, 3, 4]
n = len(nums)
tests = {
    "Pool":      (pool_with,      [range(n - t) for t in range(n)]),
    "FY":        (fy_with,        [range(i, n) for i in range(n)]),
    "InsideOut": (inside_out_with, [range(i + 1) for i in range(n)]),
}
for name, (fn, ranges) in tests.items():
    cnt = exact_counts(fn, nums, ranges)
    assert len(cnt) == factorial(n) and set(cnt.values()) == {1}
    print(f"{name:<10} {sum(cnt.values())} tổ hợp -> {len(cnt)} hoán vị khác nhau, mỗi hoán vị đúng 1 lần")

# Cách 2: mọi thứ tự khóa (hoán vị của hạng 0..n-1) cũng cho đúng mọi hoán vị 1 lần
cnt = Counter(sortkey_with(nums, ch) for ch in permutations(range(n)))
assert len(cnt) == factorial(n) and set(cnt.values()) == {1}
print("SortKey    mọi thứ tự khóa -> mỗi hoán vị đúng 1 lần")

Pool       24 tổ hợp -> 24 hoán vị khác nhau, mỗi hoán vị đúng 1 lần
FY         24 tổ hợp -> 24 hoán vị khác nhau, mỗi hoán vị đúng 1 lần
InsideOut  24 tổ hợp -> 24 hoán vị khác nhau, mỗi hoán vị đúng 1 lần
SortKey    mọi thứ tự khóa -> mỗi hoán vị đúng 1 lần


## 8. TRACE từng bước

Trace Fisher–Yates trên `[1, 2, 3, 4]` với lựa chọn `j = (2, 3, 2, 3)` (tức `j_0 = 2, j_1 = 3, j_2 = 2, j_3 = 3`):

| Bước `i` | `j` | Đổi `a[i]` ↔ `a[j]` | `a` sau bước | Phần đã cố định |
|---|---|---|---|---|
| 0 | 2 | `a[0]` ↔ `a[2]` | `[3, 2, 1, 4]` | `[3]` |
| 1 | 3 | `a[1]` ↔ `a[3]` | `[3, 4, 1, 2]` | `[3, 4]` |
| 2 | 2 | `a[2]` ↔ `a[2]` | `[3, 4, 1, 2]` | `[3, 4, 1]` |
| 3 | 3 | `a[3]` ↔ `a[3]` | `[3, 4, 1, 2]` | `[3, 4, 1, 2]` |

Kết quả `[3, 4, 1, 2]`. Hàm dưới in từng bước, cả trường hợp ngẫu nhiên có seed.

In [13]:
def trace_fy(nums, choices=None, seed=None):
    a = list(nums)
    n = len(a)
    rng = random.Random(seed)
    print(f"mảng gốc {a}")
    print(f"{'i':<3}{'j':<3}{'đổi':<12}{'a sau bước':<18}{'đã cố định'}")
    for i in range(n):
        j = choices[i] if choices else rng.randrange(i, n)
        assert i <= j < n
        a[i], a[j] = a[j], a[i]
        print(f"{i:<3}{j:<3}{f'a[{i}]<->a[{j}]':<12}{str(a):<18}{a[:i + 1]}")
    print("=>", a, "\n")
    return a


assert trace_fy([1, 2, 3, 4], choices=(2, 3, 2, 3)) == [3, 4, 1, 2]
trace_fy([1, 2, 3, 4], seed=7)
trace_fy([1, 2, 3, 4], seed=8)

mảng gốc [1, 2, 3, 4]
i  j  đổi         a sau bước        đã cố định
0  2  a[0]<->a[2] [3, 2, 1, 4]      [3]
1  3  a[1]<->a[3] [3, 4, 1, 2]      [3, 4]
2  2  a[2]<->a[2] [3, 4, 1, 2]      [3, 4, 1]
3  3  a[3]<->a[3] [3, 4, 1, 2]      [3, 4, 1, 2]
=> [3, 4, 1, 2] 

mảng gốc [1, 2, 3, 4]
i  j  đổi         a sau bước        đã cố định
0  2  a[0]<->a[2] [3, 2, 1, 4]      [3]
1  1  a[1]<->a[1] [3, 2, 1, 4]      [3, 2]
2  3  a[2]<->a[3] [3, 2, 4, 1]      [3, 2, 4]
3  3  a[3]<->a[3] [3, 2, 4, 1]      [3, 2, 4, 1]
=> [3, 2, 4, 1] 

mảng gốc [1, 2, 3, 4]
i  j  đổi         a sau bước        đã cố định
0  1  a[0]<->a[1] [2, 1, 3, 4]      [2]
1  2  a[1]<->a[2] [2, 3, 1, 4]      [2, 3]
2  3  a[2]<->a[3] [2, 3, 4, 1]      [2, 3, 4]
3  3  a[3]<->a[3] [2, 3, 4, 1]      [2, 3, 4, 1]
=> [2, 3, 4, 1] 



[2, 3, 4, 1]

Với cùng bước đầu ta cũng có thể xem `fy_with` (hàm nhận sẵn lựa chọn) cho ra đúng kết quả đã trace — để chắc rằng phiên bản vét cạn **chính là** thuật toán thật:

In [14]:
assert fy_with([1, 2, 3, 4], (2, 3, 2, 3)) == (3, 4, 1, 2)

# và bản thật dùng cùng công thức: ép randrange trả về dãy cho trước
class FixedRandom:
    def __init__(self, seq): self.seq = iter(seq)
    def randrange(self, a, b):
        v = next(self.seq); assert a <= v < b; return v

fr = FixedRandom((2, 3, 2, 3))
a = [1, 2, 3, 4]
for i in range(4):
    j = fr.randrange(i, 4)
    a[i], a[j] = a[j], a[i]
print(a)
assert a == [3, 4, 1, 2]

[3, 4, 1, 2]


## 9. Code nộp LeetCode

Phiên bản cuối (Cách 3 — Fisher–Yates), đúng tên lớp/method/signature:

In [15]:
class Solution:

    def __init__(self, nums: List[int]):
        self.original = nums[:]            # bản sao: người gọi sửa nums cũng không ảnh hưởng

    def reset(self) -> List[int]:
        return self.original[:]

    def shuffle(self) -> List[int]:
        a = self.original[:]
        n = len(a)
        for i in range(n):
            j = random.randrange(i, n)
            a[i], a[j] = a[j], a[i]
        return a


random.seed(10)
print(run_ops(Solution, ops, args))

[None, [3, 2, 1], [1, 2, 3], [2, 1, 3]]


## 10. Vì sao thuật toán đúng

- **Là hoán vị:** chỉ dùng các phép **đổi chỗ**, nên tập phần tử không đổi.
- **`reset` đúng:** ta giữ `self.original` (một bản sao) và **không bao giờ sửa** nó; `shuffle` làm việc trên bản sao mới mỗi lần, `reset` trả một bản sao mới.
- **Đều (quy nạp trên số phần tử còn lại):** ở bước `i`, phần tử đặt vào vị trí `i` là một phần tử **bất kỳ** của `a[i..n-1]` với xác suất `1/(n-i)`. Chuỗi lựa chọn `(j_0, …, j_{n-1})` có tổng `n · (n-1) ⋯ 1 = n!` khả năng, **mỗi khả năng đồng xác suất `1/n!`** và **mỗi khả năng cho một hoán vị khác nhau** (hai chuỗi khác nhau tại bước đầu tiên chúng khác thì phần tử ở vị trí đó khác, và các vị trí sau không "sửa" được vị trí đã cố định). Hai điều này cho `n!` hoán vị mỗi hoán vị đúng một chuỗi ⇒ xác suất `1/n!`. Kiểm chứng bằng vét cạn ở mục 7 (`n = 4`) và mục 12 (`n ≤ 6`).
- **Độc lập các lần gọi:** `shuffle` luôn xuất phát từ `original`, không phụ thuộc kết quả trước.

## 11. Phân tích Time / Space Complexity

| Cách | `shuffle` Time | Space | Ghi chú |
|---|---|---|---|
| 1. Bốc từ túi | `O(n²)` | `O(n)` | `pop` giữa list |
| 2. Sắp xếp theo khóa | `O(n log n)` | `O(n)` | sắp xếp tốn log |
| 3. Fisher–Yates | `O(n)` | `O(n)` (bản sao trả về) | nhanh nhất, đơn giản |
| 4. Inside-out | `O(n)` | `O(n)` | không sửa mảng gốc, hợp với luồng dữ liệu |

`__init__` và `reset`: `O(n)` (sao chép). Với `n ≤ 50` và `10^4` lần gọi, mọi cách đều thừa nhanh; điểm của bài là **tính đều** chứ không phải tốc độ.

## 12. Bộ test

Chiến lược (bài ngẫu nhiên: **tính chất + thống kê có seed + vét cạn**):

1. **Ví dụ đề:** `shuffle` là hoán vị của `[1, 2, 3]`, `reset` trả đúng `[1, 2, 3]`.
2. **Tính chất:** mọi lần `shuffle`/`reset` trả đúng độ dài, đủ phần tử; `reset` sau nhiều lần `shuffle` vẫn đúng; sửa list trả về **không** làm hỏng lần sau; sửa `nums` của người gọi không ảnh hưởng.
3. **Edge case:** một phần tử, hai phần tử, số âm, `n = 50`.
4. **Thống kê:** `n = 3, 4` ⇒ mọi hoán vị xuất hiện ≈ `trials / n!` (≤ 5σ), và mỗi phần tử ở mỗi vị trí với xác suất `1/n`.
5. **Vét cạn:** mục 7; thêm `n = 5, 6` cho Fisher–Yates.

In [16]:
IMPLS = [SolutionPool, SolutionSortKey, SolutionFY, SolutionInsideOut, Solution]


def check_uniform_counts(counts, keys, trials):
    k = len(keys)
    p = 1 / k
    sigma = (trials * p * (1 - p)) ** 0.5 if k > 1 else 1
    assert set(counts) <= set(keys), "xuất hiện kết quả lạ"
    worst = max(abs(counts.get(key, 0) - trials * p) / sigma for key in keys)
    assert worst < 5, worst
    return worst


# 1) Ví dụ đề
for cls in IMPLS:
    random.seed(5)
    r = run_ops(cls, ops, args)
    assert r[0] is None
    assert sorted(r[1]) == [1, 2, 3] and r[2] == [1, 2, 3] and sorted(r[3]) == [1, 2, 3]
print("Ví dụ đề: OK")

# 2) Tính chất: reset, độc lập, không alias
for cls in IMPLS:
    random.seed(6)
    src = [5, -3, 9, 0, 12]
    s = cls(src)
    for _ in range(100):
        a = s.shuffle()
        assert sorted(a) == sorted(src) and len(a) == len(src)
        a[0] = 10**9                          # phá list trả về
        assert s.reset() == [5, -3, 9, 0, 12]
        b = s.reset()
        b.append(777)                         # phá list reset
        assert s.reset() == [5, -3, 9, 0, 12]
    src[0] = 12345                            # người gọi sửa mảng gốc
    assert s.reset() == [5, -3, 9, 0, 12], "phải giữ bản sao, không giữ alias"
print("Tính chất / không alias: OK")

Ví dụ đề: OK
Tính chất / không alias: OK


In [17]:
# 3) Edge cases
for cls in IMPLS:
    random.seed(7)
    assert cls([42]).shuffle() == [42] and cls([42]).reset() == [42]
    assert {tuple(cls([1, 2]).shuffle()) for _ in range(100)} == {(1, 2), (2, 1)}
    assert sorted(cls(list(range(-25, 25))).shuffle()) == list(range(-25, 25))   # n = 50
print("Edge cases: OK")

# 4) Thống kê: mọi hoán vị đều xuất hiện ~ trials / n!
for n, trials in [(3, 60000), (4, 120000)]:
    base = list(range(n))
    allperms = list(permutations(base))
    for cls in IMPLS:
        random.seed(100 + n)
        s = cls(base)
        cnt = Counter(tuple(s.shuffle()) for _ in range(trials))
        worst = check_uniform_counts(cnt, allperms, trials)
    print(f"n={n}: {len(allperms)} hoán vị, {trials} lần; cả {len(IMPLS)} cách đều đều (cách cuối lệch {worst:.2f} sigma)")

Edge cases: OK


n=3: 6 hoán vị, 60000 lần; cả 5 cách đều đều (cách cuối lệch 1.07 sigma)


n=4: 24 hoán vị, 120000 lần; cả 5 cách đều đều (cách cuối lệch 1.55 sigma)


In [18]:
# 4b) Ma trận vị trí: phần tử x ở vị trí p với xác suất 1/n (n = 6, kiểm tra từng ô)
n, trials = 6, 60000
for cls in IMPLS:
    random.seed(2025)
    s = cls(list(range(n)))
    pos_cnt = [Counter() for _ in range(n)]
    for _ in range(trials):
        for p, x in enumerate(s.shuffle()):
            pos_cnt[p][x] += 1
    worst = max(check_uniform_counts(pos_cnt[p], range(n), trials) for p in range(n))
    print(f"{cls.__name__:<18} ma trận vị trí đều, lệch tối đa {worst:.2f} sigma")

# 5) Vét cạn thêm cho Fisher–Yates & inside-out với n = 5, 6
for n in (5, 6):
    nums = list(range(n))
    for name, fn, ranges in [("FY", fy_with, [range(i, n) for i in range(n)]),
                             ("InsideOut", inside_out_with, [range(i + 1) for i in range(n)])]:
        cnt = exact_counts(fn, nums, ranges)
        assert len(cnt) == factorial(n) and set(cnt.values()) == {1}
    print(f"n={n}: vét cạn {factorial(n)} hoán vị, mỗi hoán vị đúng 1 chuỗi lựa chọn")

SolutionPool       ma trận vị trí đều, lệch tối đa 1.87 sigma


SolutionSortKey    ma trận vị trí đều, lệch tối đa 1.97 sigma


SolutionFY         ma trận vị trí đều, lệch tối đa 1.87 sigma


SolutionInsideOut  ma trận vị trí đều, lệch tối đa 2.31 sigma


Solution           ma trận vị trí đều, lệch tối đa 1.87 sigma
n=5: vét cạn 120 hoán vị, mỗi hoán vị đúng 1 chuỗi lựa chọn
n=6: vét cạn 720 hoán vị, mỗi hoán vị đúng 1 chuỗi lựa chọn


In [19]:
# 6) So với thư viện: random.shuffle (cũng là Fisher-Yates) và mọi cách có cùng phân phối
random.seed(77)
base = [1, 2, 3, 4]
lib = Counter()
for _ in range(48000):
    a = base[:]; random.shuffle(a); lib[tuple(a)] += 1
check_uniform_counts(lib, list(permutations(base)), 48000)
print("random.shuffle của thư viện cũng đều: OK")

# 7) Hiệu năng (chỉ in): 10^4 lần gọi, n = 50
base = list(range(50))
for cls in IMPLS:
    random.seed(0)
    s = cls(base)
    t = time.perf_counter()
    for _ in range(10000):
        s.shuffle()
    print(f"{cls.__name__:<18} {time.perf_counter() - t:.3f}s")

random.shuffle của thư viện cũng đều: OK


SolutionPool       0.177s


SolutionSortKey    0.097s


SolutionFY         0.127s


SolutionInsideOut  0.131s


Solution           0.116s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đổi chỗ với **mọi** vị trí `[0, n-1]` ở mỗi bước (thiên vị)

Viết `j = random.randrange(n)` thay vì `randrange(i, n)` trông rất giống nhưng cho `n^n` kết quả đồng khả năng — **không chia hết** cho `n!` ⇒ **không thể đều**. Với `n = 3`: `27` kết quả trên `6` hoán vị. Đếm chính xác:

In [20]:
def naive_with(nums, ch):             # SAI: ch[i] ∈ [0, n-1] cho mọi i
    a = list(nums)
    for i, j in enumerate(ch):
        a[i], a[j] = a[j], a[i]
    return tuple(a)


n = 3
cnt = exact_counts(naive_with, [1, 2, 3], [range(n)] * n)
print("tổng khả năng:", sum(cnt.values()), "(27 không chia hết cho 6)")
for p, c in sorted(cnt.items()):
    print(p, c, f"-> xác suất {c}/27")

tổng khả năng: 27 (27 không chia hết cho 6)
(1, 2, 3) 4 -> xác suất 4/27
(1, 3, 2) 5 -> xác suất 5/27
(2, 1, 3) 5 -> xác suất 5/27
(2, 3, 1) 5 -> xác suất 5/27
(3, 1, 2) 4 -> xác suất 4/27
(3, 2, 1) 4 -> xác suất 4/27


Có hoán vị xuất hiện `5/27`, có hoán vị chỉ `4/27` (thay vì `1/6 ≈ 4.5/27`). **Sửa:** `j = randrange(i, n)` (chỉ chọn trong phần chưa cố định).

### Lỗi 2: sai biên thành thuật toán Sattolo (không bao giờ giữ nguyên vị trí)

Chọn `j` trong `[i+1, n-1]` (hoặc `[0, i-1]` kiểu ngược) loại trừ `j == i` ⇒ **không phần tử nào ở yên** và ta chỉ nhận được **`(n-1)!` hoán vị** (các hoán vị vòng đơn). Hãy đếm cho `n = 4`:

In [21]:
def sattolo_with(nums, ch):           # j ∈ [i+1, n-1]  (loại trừ i)
    a = list(nums)
    for i, j in enumerate(ch):
        a[i], a[j] = a[j], a[i]
    return tuple(a)


n = 4
ranges = [range(i + 1, n) for i in range(n - 1)]      # bước cuối bị bỏ vì không còn lựa chọn
cnt = exact_counts(sattolo_with, [0, 1, 2, 3], ranges)
print("số hoán vị xuất hiện:", len(cnt), "trên", factorial(n), "(chỉ (n-1)! =", factorial(n - 1), ")")
print("có hoán vị giữ nguyên [0,1,2,3] không?", (0, 1, 2, 3) in cnt)
assert len(cnt) == factorial(n - 1) and (0, 1, 2, 3) not in cnt

số hoán vị xuất hiện: 6 trên 24 (chỉ (n-1)! = 6 )
có hoán vị giữ nguyên [0,1,2,3] không? False


Một người dùng sẽ **không bao giờ** thấy mảng giữ nguyên hay có phần tử đứng yên — không phải trộn đều. **Sửa:** cho phép `j == i` (`randrange(i, n)`).

### Lỗi 3: trộn tại chỗ trên mảng gốc ⇒ `reset` không khôi phục được

Nếu `shuffle` sửa trực tiếp `self.nums` (và không giữ bản gốc) thì `reset()` không biết mảng gốc là gì. Tương tự, `self.nums = nums` chỉ **alias** mảng của người gọi.

In [22]:
class BadInPlace:
    def __init__(self, nums):
        self.nums = nums                    # alias, không sao chép
    def reset(self):
        return self.nums                    # trả mảng đã bị trộn!
    def shuffle(self):
        n = len(self.nums)
        for i in range(n):
            j = random.randrange(i, n)
            self.nums[i], self.nums[j] = self.nums[j], self.nums[i]
        return self.nums

random.seed(3)
b = BadInPlace([1, 2, 3, 4, 5])
b.shuffle()
print("reset trả:", b.reset(), "(đề muốn [1, 2, 3, 4, 5])")

reset trả: [2, 3, 4, 5, 1] (đề muốn [1, 2, 3, 4, 5])


**Sửa:** giữ `self.original = nums[:]` bất biến, trộn trên bản sao.

### Lỗi 4: trả về chính `self.original` (hoặc cùng một list) ⇒ người gọi làm hỏng trạng thái

`reset()` mà `return self.original` thì nếu người gọi sửa list trả về, mảng gốc bị đổi theo. An toàn nhất là trả **bản sao** (`self.original[:]`) — test mục 12 đã bắt lỗi này.

### Lỗi 5: sắp xếp bằng hàm so sánh ngẫu nhiên

`sorted(nums, key=cmp_to_key(lambda a, b: random.choice([-1, 1])))` trông như "xáo trộn" nhưng hàm so sánh **không nhất quán** (`a < b` rồi `b < a`), thuật toán sắp xếp chỉ thực hiện **một số phép so sánh cố định** ⇒ số chuỗi kết quả là lũy thừa của 2, không thể chia đều cho `n!` (không phải lũy thừa của 2 khi `n ≥ 3`).

In [23]:
def bad_cmp_shuffle(nums):
    return sorted(nums, key=cmp_to_key(lambda a, b: random.choice([-1, 1])))


random.seed(5)
trials = 60000
cnt = Counter(tuple(bad_cmp_shuffle([1, 2, 3])) for _ in range(trials))
for p, c in sorted(cnt.items()):
    print(p, f"{c / trials:.3f}", "(đều phải là 0.167)")
try:
    check_uniform_counts(cnt, list(permutations([1, 2, 3])), trials)
except AssertionError as e:
    print("Phát hiện lệch > 5 sigma:", round(float(e.args[0]), 1))

(1, 2, 3) 0.314 (đều phải là 0.167)
(1, 3, 2) 0.031 (đều phải là 0.167)
(2, 1, 3) 0.095 (đều phải là 0.167)
(2, 3, 1) 0.141 (đều phải là 0.167)
(3, 1, 2) 0.156 (đều phải là 0.167)
(3, 2, 1) 0.263 (đều phải là 0.167)
Phát hiện lệch > 5 sigma: 97.0


**Sửa:** dùng khóa **cố định theo phần tử** (`key=`/danh sách cặp `(random(), x)`) như Cách 2, hoặc Fisher–Yates.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Trộn đều"** ⇒ **Fisher–Yates**: bước `i` chọn `j ∈ [i, n-1]` rồi đổi chỗ. Thuộc lòng biên **`randrange(i, n)`**.
- Kiểm tra "có đều không" bằng **đếm số chuỗi lựa chọn**: phải là `n!` (hoặc bội đồng đều); `n^n` thì **sai**.
- Bài có `reset`/khôi phục ⇒ giữ **bản sao bất biến**, làm việc trên **bản sao khác**; trả bản sao để người gọi không làm hỏng.
- Có sẵn `random.shuffle(list)` (trộn tại chỗ, Fisher–Yates) — dùng trong code thật, nhưng phỏng vấn thường yêu cầu tự viết.
- Test ngẫu nhiên: **seed + vét cạn khi `n` nhỏ + thống kê ≤ 5σ**, tuyệt đối không so với một đáp án cố định.

## 15. Biến thể của bài

1. **Chọn `k` phần tử ngẫu nhiên khác nhau (không lặp):** chỉ chạy `k` bước đầu của Fisher–Yates — `O(k)` sau khi sao chép (hoặc dùng `random.sample`).
2. **LeetCode 382 — Linked List Random Node / 398 — Random Pick Index:** chọn một phần tử đều từ luồng (Reservoir Sampling).
3. **Xáo trộn không phần tử nào đứng yên (derangement):** thử Sattolo cho hoán vị vòng đơn, hoặc trộn rồi loại nếu có điểm cố định (rejection).
4. **LeetCode 519 — Random Flip Matrix / 710 — Random Pick with Blacklist:** Fisher–Yates **ảo** bằng `dict` ghi lại các phép đổi chỗ để chọn không lặp mà không lưu cả mảng.

Cài thử biến thể 1 (chọn `k`) và Sattolo (3):

In [24]:
def sample_k(nums, k):
    """Partial Fisher-Yates: k phần tử khác nhau, mỗi tập con xác suất như nhau."""
    a = list(nums)
    n = len(a)
    for i in range(k):
        j = random.randrange(i, n)
        a[i], a[j] = a[j], a[i]
    return a[:k]


random.seed(15)
from itertools import combinations
sets = Counter(frozenset(sample_k([1, 2, 3, 4, 5], 2)) for _ in range(50000))
allsets = [frozenset(c) for c in combinations([1, 2, 3, 4, 5], 2)]
print(len(sets), "tập con 2 phần tử, mỗi tập ~", 50000 // 10)
check_uniform_counts(sets, allsets, 50000)


def sattolo(nums):
    a = list(nums)
    for i in range(len(a) - 1):
        j = random.randrange(i + 1, len(a))
        a[i], a[j] = a[j], a[i]
    return a


random.seed(16)
for _ in range(500):
    s = sattolo([0, 1, 2, 3, 4])
    assert all(s[i] != i for i in range(5))      # Sattolo cho hoán vị vòng đơn: không có điểm cố định
print("Sattolo: không bao giờ có phần tử đứng yên")

10 tập con 2 phần tử, mỗi tập ~ 5000
Sattolo: không bao giờ có phần tử đứng yên


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time `shuffle` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bốc từ túi | `pop` ngẫu nhiên đến khi hết | `O(n²)` | `O(n)` | Hiểu trực giác, `n` rất nhỏ |
| 2. Sắp xếp theo khóa | mỗi phần tử một khóa ngẫu nhiên | `O(n log n)` | `O(n)` | Gọn trong 1 dòng; dùng khi cần ổn định/kết hợp sắp xếp |
| 3. Fisher–Yates | chọn `j ∈ [i, n-1]` và đổi chỗ | `O(n)` | `O(n)` | **Chuẩn**, nên dùng và nên tự viết được |
| 4. Inside-out | dựng mảng mới từng phần tử | `O(n)` | `O(n)` | Khi **không muốn sửa nguồn** hoặc nguồn là luồng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code. (a) `a = [1,2,3]; b = a; b[0] = 9; print(a)`; (b) `a = [1,2,3]; c = a[:]; c[0] = 9; print(a)`; (c) `[1,2,3,4]` sau `a[0], a[3] = a[3], a[0]`.

**Bài 2 (tính tay).** Với `[10, 20, 30]` và lựa chọn Fisher–Yates `j = (2, 1, 2)` (tức `j_0 = 2, j_1 = 1, j_2 = 2`), viết mảng sau mỗi bước. Rồi đếm: với `n = 3` có bao nhiêu chuỗi lựa chọn `(j_0, j_1, j_2)` hợp lệ (`j_i ∈ [i, 2]`)? (đáp án: 6)

**Bài 3 (điền TODO).** Hoàn thành `fisher_yates_todo(nums)` trả mảng đã trộn (không sửa `nums`).

In [25]:
def fisher_yates_todo(nums):
    # TODO 1: a = bản sao của nums
    # TODO 2: for i in range(len(a)): chọn j = random.randrange(i, len(a))
    # TODO 3: đổi chỗ a[i], a[j] = a[j], a[i]
    # TODO 4: trả a
    return None

# KHÔNG gọi fisher_yates_todo ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `class SolutionInsideOut2` theo Cách 4 (inside-out) **mà không nhìn lại** mục 7; kiểm tra bằng `check_uniform_counts` trên `n = 4` và vét cạn bằng `inside_out_with`.

**Bài 5 (biến thể / mini challenge).** Viết `shuffle_partial(nums, k)` chỉ trộn **`k` vị trí đầu** sao cho `k` phần tử đầu là một **chỉnh hợp** (có thứ tự) ngẫu nhiên đều của `k` phần tử khác nhau trong `nums`, trong thời gian `O(k)` **sau khi sao chép**. Kiểm tra: với `nums = [1,2,3,4]`, `k = 2` có `4·3 = 12` chỉnh hợp, mỗi chỉnh hợp ~ `1/12`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `[9, 2, 3]` vì alias; (b) `[1, 2, 3]`; (c) `[4, 2, 3, 1]`. Bài 3: nhớ biên `randrange(i, n)`.
- **Mức 2 (vừa):** Bài 2: `i=0,j=2`: `[30,20,10]`; `i=1,j=1`: không đổi; `i=2,j=2`: không đổi ⇒ `[30,20,10]`. Số chuỗi: `3 × 2 × 1 = 6`. Bài 4: `j = randint(0, i)`; `if j != i: a[i] = a[j]`; `a[j] = src[i]`.
- **Mức 3 (sâu):** Bài 5: chạy đúng `k` bước đầu của Fisher–Yates (`i = 0..k-1`, `j ∈ [i, n-1]`) rồi lấy `a[:k]`: số chuỗi `n·(n-1)⋯(n-k+1)` = số chỉnh hợp `A(n, k)`, mỗi chuỗi cho một chỉnh hợp khác nhau.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
a = [1, 2, 3]; b = a; b[0] = 9
assert a == [9, 2, 3]
a = [1, 2, 3]; c = a[:]; c[0] = 9
assert a == [1, 2, 3]
x = [1, 2, 3, 4]; x[0], x[3] = x[3], x[0]
assert x == [4, 2, 3, 1]

# Bài 2
assert fy_with([10, 20, 30], (2, 1, 2)) == (30, 20, 10)
assert len(list(product(*[range(i, 3) for i in range(3)]))) == 6


# Bài 3
def fisher_yates_todo_ans(nums):
    a = list(nums)
    for i in range(len(a)):
        j = random.randrange(i, len(a))
        a[i], a[j] = a[j], a[i]
    return a


src3 = [1, 2, 3]
random.seed(31)
cnt3 = Counter(tuple(fisher_yates_todo_ans(src3)) for _ in range(60000))
check_uniform_counts(cnt3, list(permutations(src3)), 60000)
assert src3 == [1, 2, 3]                      # không sửa đầu vào


# Bài 4
class SolutionInsideOut2:
    def __init__(self, nums):
        self.original = nums[:]

    def reset(self):
        return self.original[:]

    def shuffle(self):
        src = self.original
        a = [0] * len(src)
        for i in range(len(src)):
            j = random.randint(0, i)
            if j != i:
                a[i] = a[j]
            a[j] = src[i]
        return a


random.seed(41)
s4 = SolutionInsideOut2([1, 2, 3, 4])
cnt4 = Counter(tuple(s4.shuffle()) for _ in range(72000))
check_uniform_counts(cnt4, list(permutations([1, 2, 3, 4])), 72000)
assert s4.reset() == [1, 2, 3, 4]


# Bài 5
def shuffle_partial(nums, k):
    a = list(nums)
    n = len(a)
    for i in range(k):
        j = random.randrange(i, n)
        a[i], a[j] = a[j], a[i]
    return a[:k]


random.seed(51)
cnt5 = Counter(tuple(shuffle_partial([1, 2, 3, 4], 2)) for _ in range(60000))
arrangements = list(permutations([1, 2, 3, 4], 2))
assert len(arrangements) == 12
check_uniform_counts(cnt5, arrangements, 60000)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Fisher–Yates trên bản sao, giữ bản gốc bất biến)

```python
import random

class Solution:
    def __init__(self, nums: List[int]):
        self.original = nums[:]

    def reset(self) -> List[int]:
        return self.original[:]

    def shuffle(self) -> List[int]:
        a = self.original[:]
        n = len(a)
        for i in range(n):
            j = random.randrange(i, n)     # [i, n-1]
            a[i], a[j] = a[j], a[i]
        return a
```

**Checklist**

- [ ] Biên là **`randrange(i, n)`** (gồm `i`, loại `n`) — không `randrange(n)` (lệch), không `i+1` (Sattolo)?
- [ ] Giữ `self.original` là **bản sao bất biến**; `shuffle` làm việc trên **bản sao khác**?
- [ ] `reset` trả bản sao (hoặc ít nhất không để người gọi làm hỏng trạng thái)?
- [ ] Không dùng hàm so sánh ngẫu nhiên để sắp xếp?
- [ ] Đã đếm: số chuỗi lựa chọn có là `n!` (mỗi chuỗi một hoán vị)?
- [ ] Test bằng **tính chất** (là hoán vị), **vét cạn** (n nhỏ) hoặc **thống kê có seed** (≤ 5σ), không so cứng?
- [ ] Đã thử: `n = 1`, `n = 2`, số âm, `n = 50`?